# Chapter 21 Coding Exam — Leakage Auditing, Scaling Inflation & Learning Curves (100 Points)

**Book Reference:** Chapter 21, Section 21.7 (Coding Exam)  
**Suggested Time:** 40–50 minutes (Open Book, Closed Solution)

| Task | ML Paradigm, Leakage & Bias-Variance Engineering | Points |
| :--- | :--- | :---: |
| **Task 21.1** | `audit_split_leakage(train_df, test_df, id_col, group_col=None, time_col=None)` | 30 |
| **Task 21.2** | `quantify_preprocessing_leakage(X, y, seed=42)` (Leaky vs. Clean `StandardScaler` + `LogisticRegression`) | 35 |
| **Task 21.3** | `build_learning_curve_table(X, y, fractions, seed=42)` (Stratified holdout + `DecisionTreeClassifier(max_depth=4)`) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Chapter 21 Exam Grader
!pip install -q sharukh-tutor
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
import sharukh_tutor as tutor

tutor.activate(chapter="ch21", notebook="coding_exam")


---
## Exam Task 21.1 (30 Points): `audit_split_leakage(train_df, test_df, id_col, group_col=None, time_col=None)`

Write an automated split-leakage auditor that inspects `train_df` and `test_df` without mutating either DataFrame and returns a dictionary with exact keys:
1. `"id_overlap"`: integer count of unique `id_col` values present in BOTH `train_df` and `test_df`.
2. `"group_overlap"`: integer count of unique `group_col` values in both splits if `group_col is not None`, else `0`.
3. `"temporal_leak"`: `True` if `time_col is not None` and `train_df[time_col].max() >= test_df[time_col].min()`, else `False`.
4. `"is_leak_free"`: `True` only when `id_overlap == 0`, `group_overlap == 0`, and `not temporal_leak`.

In [ ]:
def audit_split_leakage(
    train_df, test_df, id_col, group_col=None, time_col=None
):
    """Audit ID, group, and temporal leakage across train/test splits."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    id_overlap = len(set(train_df[id_col]) & set(test_df[id_col]))
    group_overlap = (
        len(set(train_df[group_col]) & set(test_df[group_col]))
        if group_col is not None
        else 0
    )
    temporal_leak = (
        bool(train_df[time_col].max() >= test_df[time_col].min())
        if time_col is not None
        else False
    )
    is_leak_free = bool(
        id_overlap == 0 and group_overlap == 0 and not temporal_leak
    )
    return {
        "id_overlap": int(id_overlap),
        "group_overlap": int(group_overlap),
        "temporal_leak": temporal_leak,
        "is_leak_free": is_leak_free,
    }
    ### END SOLUTION


---
## Exam Task 21.2 (35 Points): `quantify_preprocessing_leakage(X, y, seed=42)`

Compare a **leaky** pre-split scaling workflow against a **clean** fit-on-train-only scaling workflow using `train_test_split(..., test_size=0.2, stratify=y, random_state=seed)` and `LogisticRegression(max_iter=500)`:
1. **Leaky path:** Fit `StandardScaler().fit_transform(X)` on the full matrix BEFORE calling `train_test_split`, then fit `LogisticRegression(max_iter=500)` on train and score accuracy on test (`leaky_acc`).
2. **Clean path:** Call `train_test_split` FIRST, fit `StandardScaler()` on `X_train` ONLY, transform both `X_train` and `X_test`, then fit `LogisticRegression(max_iter=500)` on train and score accuracy on test (`clean_acc`).
3. Return `{"leaky_acc": round4, "clean_acc": round4, "optimism_gap": round4}` where `optimism_gap = leaky_acc - clean_acc`.

In [ ]:
def quantify_preprocessing_leakage(X, y, seed=42):
    """Compare pre-split vs. train-only StandardScaler accuracy."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    X_arr = np.asarray(X, dtype=float)
    y_arr = np.asarray(y)

    X_all_s = StandardScaler().fit_transform(X_arr)
    Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(
        X_all_s, y_arr, test_size=0.2, stratify=y_arr, random_state=seed
    )
    leaky_acc = float(
        LogisticRegression(max_iter=500)
        .fit(Xl_tr, yl_tr)
        .score(Xl_te, yl_te)
    )

    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
        X_arr, y_arr, test_size=0.2, stratify=y_arr, random_state=seed
    )
    scaler = StandardScaler().fit(Xc_tr)
    clean_acc = float(
        LogisticRegression(max_iter=500)
        .fit(scaler.transform(Xc_tr), yc_tr)
        .score(scaler.transform(Xc_te), yc_te)
    )

    return {
        "leaky_acc": round(leaky_acc, 4),
        "clean_acc": round(clean_acc, 4),
        "optimism_gap": round(leaky_acc - clean_acc, 4),
    }
    ### END SOLUTION


---
## Exam Task 21.3 (35 Points): `build_learning_curve_table(X, y, fractions, seed=42)`

Construct a bias-variance learning curve DataFrame across increasing training fractions:
1. Hold out a 25% stratified validation set via `train_test_split(np.asarray(X, dtype=float), np.asarray(y), test_size=0.25, stratify=y, random_state=seed)`.
2. For each `frac` in `fractions`, slice the first `n_train = int(round(frac * len(X_train_full)))` rows of the shuffled training split.
3. Train `DecisionTreeClassifier(max_depth=4, random_state=seed)` on that training subset and evaluate `train_acc`, `val_acc`, and `gap = train_acc - val_acc`.
4. Return a `pd.DataFrame` with exact columns `["fraction", "n_train", "train_acc", "val_acc", "gap"]`, with float columns rounded to **4 decimal places**.

In [ ]:
def build_learning_curve_table(X, y, fractions, seed=42):
    """Evaluate DecisionTreeClassifier(max_depth=4) across train sizes."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    X_arr = np.asarray(X, dtype=float)
    y_arr = np.asarray(y)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_arr, y_arr, test_size=0.25, stratify=y_arr, random_state=seed
    )
    rows = []
    for frac in fractions:
        n_sub = int(round(frac * len(X_tr)))
        clf = DecisionTreeClassifier(max_depth=4, random_state=seed)
        clf.fit(X_tr[:n_sub], y_tr[:n_sub])
        tr_acc = float(clf.score(X_tr[:n_sub], y_tr[:n_sub]))
        val_acc = float(clf.score(X_val, y_val))
        rows.append({
            "fraction": round(float(frac), 4),
            "n_train": int(n_sub),
            "train_acc": round(tr_acc, 4),
            "val_acc": round(val_acc, 4),
            "gap": round(tr_acc - val_acc, 4),
        })
    cols = ["fraction", "n_train", "train_acc", "val_acc", "gap"]
    return pd.DataFrame(rows, columns=cols)
    ### END SOLUTION


In [ ]:
exam_submissions = {
    "ch21_exam_01": audit_split_leakage,
    "ch21_exam_02": quantify_preprocessing_leakage,
    "ch21_exam_03": build_learning_curve_table,
}

tutor.check_exam("ch21", exam_submissions)


---
## Chapter 21 Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 21.1 (`>=` Temporal Boundary):** If `train_df['year'].max() == 2024` and `test_df['year'].min() == 2024`, records from the same epidemic year straddle both sets. Using `>=` (not strict `>`) catches both same-period overlap and future-in-past reversals.
- **Task 21.2 (Why Scaling Before Splitting Leaks):** Even an unsupervised `StandardScaler` computes $\mu$ and $\sigma$ across all rows. When high-dimensional or unevenly scaled features are standardized before splitting, test-set outliers shift the training geometry and inflate logistic regression accuracy (`+0.0833` on our 30-feature clinical cohort).
- **Task 21.3 (Slicing Instead of `train_test_split(train_size=1.0)`):** Calling `train_test_split(..., train_size=1.0)` raises a `ValueError` in Scikit-Learn. Because the initial 75/25 `train_test_split` already shuffles the training pool deterministically, slicing `X_tr[:n_sub]` handles every fraction—including `1.0`—cleanly.